# Browse size-dependent ablation results

**Role: analysis / interactive display of saved results.** Explore the already-computed N-dependent ablation responses for one center identity at a time. Select several perturbed neighbor identities and compare their curves within a panel for each exact hop distance. The center identity is held fixed; the curve labels identify the neighboring fate that was perturbed.

This notebook reads `size_summary.csv` and the accompanying settings from a completed `size_dependent_ablation.ipynb` analysis. It performs no training, inference, resampling, or new normalization. Fold/seed aggregation, size-bin weighting and uncertainty are those saved by the generating analysis. The default directory contains the 5% masking results and all three curvature scales, including the unmasked depth-0 reference.

**Interpretation:** effects are edited minus intact predictions. Observed curves group original organoids by their observed N; sweeps vary the supplied N on fixed sampled graphs. Different marker pairs can have different sampled neighborhoods, so these overlays do not by themselves form a matched comparison between markers. Bands are saved pointwise organoid-bootstrap intervals, conditional on fitted models. Near-zero depth-0 reference exclusions were applied upstream; they can change support relative to other normalizations.

In [ ]:
from pathlib import Path
import json
import html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib import ticker as mticker
import ipywidgets as widgets
from IPython.display import display, clear_output
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/models/gnn.py').is_file())  # Find the repository from either working directory.


## Analysis settings

Point `RESULTS_DIR` to the directory containing the summary you want to browse, including the normalization subdirectory when using depth-0-normalized results. This is an explicit result selection; the notebook never chooses the latest run automatically. Defaults below only initialize the controls. Support limits inherit the generating notebook's settings unless overridden.

In [ ]:
# Saved results
RESULTS_DIR = PROJECT_ROOT / 'training_results/fate_masking_training/Ndependent_20260922_115100/analysis/size_dependent_ablation/film_d2_h128_masking_p0.05/masking_5pct_size_stratified/depth0_normalization_3e408daab068'  # Directory containing size_summary.csv.

# Initial widget selection
DEFAULT_CENTER = 'LGR5'  # Center identity; fall back to the first available identity if absent.
DEFAULT_PERTURBATIONS = ['KI67', 'Lysozyme', 'Serotonin']  # Initially selected neighboring identities.
DEFAULT_VIEW = 'sweep'  # sweep, observed, or both.
DEFAULT_METRIC = 'delta_depth0_relative'  # delta_mu, delta_relative, or delta_depth0_relative, when saved.
SHOW_INTERVALS = True  # Initially display saved bootstrap bands.

# Figure layout and display support
LOG_N = True  # Use a logarithmic N axis.
PANEL_COLUMNS = 2  # Maximum number of hop panels in a row.
SHARE_Y = True  # Give hop panels the same vertical scale for magnitude comparisons.
MIN_CASES = None  # None inherits the saved minimum number of unique physical cases.
MIN_ORGANOIDS = None  # None inherits the saved minimum number of organoids.


## Load the saved summaries

Read the plotted means, intervals, support counts and provenance. Available center identities, perturbation identities, hops and normalizations are derived from the saved table. Only saved scales are offered; switching normalization selects a different saved summary, rather than dividing already-averaged curves.

In [ ]:
summary_path = RESULTS_DIR / 'size_summary.csv'
if not summary_path.is_file():
    raise FileNotFoundError(f'No saved size summary at {summary_path}. Run the generating analysis first.')
summary = pd.read_csv(summary_path)
required = {'metric', 'analysis', 'center_marker', 'source_marker_name', 'hop', 'coordinate',
            'N', 'mean', 'ci_low', 'ci_high', 'n_cases', 'n_organoids'}
if missing := required - set(summary):
    raise ValueError(f'Saved summary lacks required columns: {sorted(missing)}')
if summary.empty:
    raise ValueError('The saved size summary is empty.')
for column in ['hop', 'coordinate', 'N', 'mean', 'ci_low', 'ci_high', 'n_cases', 'n_organoids']:
    summary[column] = pd.to_numeric(summary[column], errors='raise')
if not np.isfinite(summary.N).all() or not summary.N.gt(0).all():
    raise ValueError('The saved N coordinates must be finite and positive.')
if not summary.hop.ge(1).all() or not summary.hop.mod(1).eq(0).all():
    raise ValueError('Expected positive integer hop distances.')
summary['hop'] = summary.hop.astype(int)
keys = ['metric', 'analysis', 'center_marker', 'source_marker_name', 'hop', 'coordinate']
if summary.duplicated(keys).any():
    raise ValueError('This viewer expects one aggregated result per pair/hop/N. Select a single-method summary.')
if not set(summary.analysis) <= {'observed', 'sweep'}:
    raise ValueError('Expected observed and/or sweep rows.')
settings_path = RESULTS_DIR / 'settings.json'
result_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
plot_path = RESULTS_DIR / 'plot_settings.json'
saved_plot_settings = json.loads(plot_path.read_text()) if plot_path.exists() else {}
source_path = Path(result_settings.get('ablation_directory', RESULTS_DIR)) / 'settings.json'
source_settings = json.loads(source_path.read_text()) if source_path.exists() else result_settings
case_threshold = saved_plot_settings.get('heatmap_display_min_cases', 1) if MIN_CASES is None else MIN_CASES
organoid_threshold = saved_plot_settings.get('min_plot_organoids', 1) if MIN_ORGANOIDS is None else MIN_ORGANOIDS
threshold_values = list(case_threshold.values()) if isinstance(case_threshold, dict) else [case_threshold]
if case_threshold is None:
    case_threshold, threshold_values = 0, [0]
if any(not np.isfinite(value) or value < 0 for value in [*threshold_values, organoid_threshold]):
    raise ValueError('Support thresholds must be finite and nonnegative.')
if not isinstance(PANEL_COLUMNS, int) or PANEL_COLUMNS < 1:
    raise ValueError('PANEL_COLUMNS must be a positive integer.')
METRIC_LABELS = {
    'delta_mu': ('Raw curvature change', 'ΔK (original curvature units)'),
    'delta_relative': ('Sphere-normalized', 'ΔK / Kref(N) (dimensionless)'),
    'delta_depth0_relative': ('Depth-0-normalized', 'ΔK / |K₀| (dimensionless)'),
    'delta_z': ('Transformed curvature change', 'Δ transformed curvature'),
}
metrics = [metric for metric in METRIC_LABELS if metric in set(summary.metric)]
if not metrics:
    raise ValueError('No recognized curvature metric in this summary.')
centers = sorted(summary.center_marker.unique())
perturbations = sorted(summary.source_marker_name.unique())
hops = sorted(summary.hop.unique())
views = [mode for mode in ['sweep', 'observed'] if mode in set(summary.analysis)]
if len(views) == 2:
    views.append('both')
marker_colors = dict(zip(perturbations, plt.get_cmap('tab20' if len(perturbations) > 10 else 'tab10').colors))
print('Results:', RESULTS_DIR)
print(f"Method: {source_settings.get('method', 'unspecified')}; model depth: {source_settings.get('depth', 'unspecified')}; "
      f"saved checkpoints: {len(source_settings.get('model_keys', []))}; hops: {hops}")
print('Available scales:', ', '.join(METRIC_LABELS[metric][0] for metric in metrics))
print(f'Minimum support: {case_threshold} cases; {organoid_threshold} organoids.')


## Interactive figure

Choose the **center marker** (the row of the original grid) and any number of **perturbation markers**. Use Ctrl/Cmd-click or Shift-click in the marker list to select multiple entries. Switch between sweep, observed, or both, and choose a normalization. Marker colors remain consistent across hops and controls; in the combined view, observed curves use a lighter color and solid circles, while sweeps use darker dashed squares.

### Plotting and update behavior

Each panel shows one exact hop. Unsupported or absent N bins remain gaps; curves are never interpolated across them. Clearing the marker selection gives an instruction rather than an empty plot. The output area is replaced on each change, and figures are closed after display to avoid duplicate plots. Raw units, sphere-relative units and depth-0-relative units follow the same definitions as the generating notebook. The depth-0 denominator includes the restored baseline, held fixed per original organoid during sweeps.

In [ ]:
def plot_selected_curves(center, sources, view, metric, show_intervals):
    """Overlay selected saved pair curves, retaining support gaps and hop identity."""
    selected_modes = ['observed', 'sweep'] if view == 'both' else [view]
    metric_rows = summary[summary.metric.eq(metric)]
    ncols = min(PANEL_COLUMNS, len(hops))
    nrows = int(np.ceil(len(hops) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.2*ncols, 4.6*nrows),
                             sharex=True, sharey=SHARE_Y, squeeze=False, layout='constrained')
    visible_points, total_points = 0, 0
    for ax, hop in zip(axes.ravel(), hops):
        threshold = case_threshold.get(str(hop), case_threshold.get(hop, 0)) if isinstance(case_threshold, dict) else case_threshold
        has_visible = False
        for source in sources:
            for mode in selected_modes:
                mode_rows = metric_rows[metric_rows.analysis.eq(mode)]
                curve = mode_rows[mode_rows.center_marker.eq(center) & mode_rows.source_marker_name.eq(source)
                                  & mode_rows.hop.eq(hop)].sort_values('coordinate').copy()
                if curve.empty:
                    continue
                supported = (curve.n_cases.ge(threshold) & curve.n_organoids.ge(organoid_threshold)
                             & np.isfinite(curve['mean']))
                visible_points += int(supported.sum())
                total_points += len(curve)
                has_visible |= bool(supported.any())
                curve.loc[~supported, ['mean', 'ci_low', 'ci_high']] = np.nan
                # Reindex before plotting to break paths at absent bins too.
                grid = (np.arange(int(mode_rows.coordinate.max())+1) if mode == 'observed'
                        else np.sort(mode_rows.coordinate.unique()))
                curve = curve.set_index('coordinate').reindex(grid)
                base = np.array(mcolors.to_rgb(marker_colors[source]))
                color = tuple(.7*base + .3) if view == 'both' and mode == 'observed' else tuple(base)
                label = f'{source} ({mode})' if view == 'both' else source
                ax.plot(curve.N, curve['mean'], 'o-' if mode == 'observed' else 's--',
                        color=color, label=label, linewidth=1.8, markersize=5)
                if show_intervals:
                    ax.fill_between(curve.N, curve.ci_low, curve.ci_high, color=color, alpha=.14)
        ax.axhline(0, color='grey', linewidth=.8)
        ax.set(title=f'Hop {hop}', xlabel='Observed cell count N' if view == 'observed' else 'Cell count N',
               ylabel=METRIC_LABELS[metric][1], xscale='log' if LOG_N else 'linear')
        if LOG_N:
            ax.xaxis.set_major_locator(mticker.LogLocator(base=10, subs=(1., 2., 5.)))
            ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
            ax.xaxis.set_minor_formatter(mticker.NullFormatter())
        ax.grid(alpha=.18)
        if not has_visible:
            ax.text(.5, .5, 'No points meet the support thresholds', ha='center', va='center', transform=ax.transAxes)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(handles, labels, fontsize=9, loc='best')
    for ax in axes.ravel()[len(hops):]:
        ax.set_visible(False)
    coordinates = metric_rows[metric_rows.analysis.isin(selected_modes)].N
    axes[0, 0].set_xlim(coordinates.min()*.9, coordinates.max()*1.1)
    fig.suptitle(f'{center} center — {source_settings.get("method", "ablation")} — '
                 f'{view.capitalize()} — {METRIC_LABELS[metric][0]}', fontsize=13)
    return fig, visible_points, total_points


def make_ablation_browser():
    """Build one independently refreshable viewer without accumulating callbacks."""
    initial_sources = tuple(marker for marker in DEFAULT_PERTURBATIONS if marker in perturbations)
    if not initial_sources:
        initial_sources = tuple(perturbations[:min(3, len(perturbations))])
    controls = {
        'center': widgets.Dropdown(options=centers, value=DEFAULT_CENTER if DEFAULT_CENTER in centers else centers[0], description='Center:'),
        'sources': widgets.SelectMultiple(options=perturbations, value=initial_sources, rows=min(10, len(perturbations)),
                                          description='Perturb:', layout=widgets.Layout(width='300px')),
        'view': widgets.ToggleButtons(options=[(mode.capitalize(), mode) for mode in views],
                                     value=DEFAULT_VIEW if DEFAULT_VIEW in views else views[0], description='View:'),
        'metric': widgets.Dropdown(options=[(METRIC_LABELS[metric][0], metric) for metric in metrics],
                                  value=DEFAULT_METRIC if DEFAULT_METRIC in metrics else metrics[0], description='Scale:',
                                  layout=widgets.Layout(width='340px')),
        'intervals': widgets.Checkbox(value=SHOW_INTERVALS, description='Show bootstrap intervals'),
    }
    output = widgets.Output()
    status = widgets.HTML()
    def refresh(change=None):
        with output:
            clear_output(wait=bool(controls['sources'].value))
            if not controls['sources'].value:
                status.value = 'Select at least one perturbation marker.'
                return
            fig, shown, total = plot_selected_curves(controls['center'].value, controls['sources'].value,
                controls['view'].value, controls['metric'].value, controls['intervals'].value)
            try:
                display(fig)
            finally:
                plt.close(fig)
        status.value = (f'{shown} of {total} saved pair/hop/N points meet display support. '
                        'Missing and unsupported points remain gaps. '
                        f'Saved sweep weighting: {html.escape(str(saved_plot_settings.get("sweep_weighting", "see source settings")))}.')
    for control in controls.values():
        control.observe(refresh, names='value')
    selectors = widgets.HBox([widgets.VBox([controls['center'], controls['view'], controls['metric'], controls['intervals']]),
                             controls['sources']], layout=widgets.Layout(flex_flow='row wrap'))
    browser = widgets.VBox([selectors, status, output])
    return browser, controls, refresh


### Display the browser

Run this cell to activate the controls. When rerunning it, the previous controls and their observers are closed first. To load another analysis, change `RESULTS_DIR` near the top and rerun the notebook.

In [ ]:
if 'ablation_controls' in globals():
    for control in ablation_controls.values():
        control.unobserve_all()
        control.close()
    for child in ablation_browser.children:
        child.close()
    ablation_browser.close()
ablation_browser, ablation_controls, refresh_ablation = make_ablation_browser()
display(ablation_browser)
refresh_ablation()
